# Exploración y KPIs — Madrid Pádel
Borja Ibáñez Pérez · ICAI

Ejecutar de arriba abajo. La carga de ambos CSV y la normalización preceden
ahora a todos los resúmenes por centro. Los nombres originales se conservan
para justificar las equivalencias. Esta copia revisada se entrega sin salidas:
regenera los resultados con los CSV de `Datos/procesados/` de tu proyecto.

In [ ]:
from pathlib import Path
import pandas as pd

carpeta_datos = Path("Datos/procesados")
carpeta_analisis = Path("Datos/analisis")
carpeta_analisis.mkdir(parents=True, exist_ok=True)

archivo = carpeta_datos / "reservas_individuales_amplias.csv"

reservas = pd.read_csv(
    archivo,
    encoding="utf-8-sig",
    parse_dates=["fecha"]
)

reservas["anio"] = reservas["fecha"].dt.year.astype("Int64")
reservas["mes"] = reservas["fecha"].dt.to_period("M").astype("string")

print("Registros:", len(reservas))
print("Nombres de centro antes de normalizar:", reservas["Centro deportivo"].nunique())

print("\nReservas por año:")
print(reservas.groupby("anio").size())

print("\nPrimera fecha:", reservas["fecha"].min())
print("Última fecha:", reservas["fecha"].max())

In [ ]:
# Cargar todos los tipos de uso de pádel aceptados y validados.
usos_padel = pd.read_csv(
    carpeta_datos / "padel_validos.csv",
    encoding="utf-8-sig",
    parse_dates=["fecha"]
)



In [ ]:
# Unificar el formato y las variantes de nombres de centros.
# Conservamos el nombre original para documentar los cambios.

equivalencias_centros = {
    "CDM El Quijote": "El Quijote"
}

for tabla in [reservas, usos_padel]:

    if "centro_nombre_original" not in tabla.columns:
        tabla["centro_nombre_original"] = tabla["Centro deportivo"]

    tabla["Centro deportivo"] = (
        tabla["centro_nombre_original"]
        .astype("string")
        .str.normalize("NFC")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace(equivalencias_centros)
    )

# Mostrar los nombres que han cambiado
cambios_nombres = (
    usos_padel[["centro_nombre_original", "Centro deportivo"]]
    .drop_duplicates()
)

cambios_nombres = cambios_nombres.loc[
    cambios_nombres["centro_nombre_original"]
    != cambios_nombres["Centro deportivo"]
].copy()

cambios_nombres["original_visible"] = (
    cambios_nombres["centro_nombre_original"].map(repr)
)

display(
    cambios_nombres[["original_visible", "Centro deportivo"]]
)

print("Nombres distintos antes:",
      reservas["centro_nombre_original"].nunique())

print("Nombres distintos después:",
      reservas["Centro deportivo"].nunique())

print("Registros individuales conservados:", len(reservas))

# Guardar las equivalencias aplicadas
cambios_nombres[
    ["centro_nombre_original", "Centro deportivo"]
].to_csv(
    carpeta_analisis / "normalizacion_centros.csv",
    index=False,
    encoding="utf-8-sig"
)

Se normalizaron los nombres de los centros eliminando espacios sobrantes y unificando «CDM El Quijote» como «El Quijote».
Esta equivalencia se adoptó al observar unidades deportivas coincidentes y periodos mensuales complementarios entre ambas denominaciones. Se conservaron los nombres originales y no se eliminaron registros por esta operación.

## Cobertura mensual tras normalizar

In [ ]:
cobertura_mensual = (
    reservas.groupby(
        ["Centro deportivo", "mes"],
        as_index=False
    )
    .agg(
        reservas=("fecha", "size"),
        dias_con_registros=("fecha", "nunique"),
        primera_fecha=("fecha", "min"),
        ultima_fecha=("fecha", "max")
    )
    .sort_values(["Centro deportivo", "mes"])
)

display(cobertura_mensual.head(20))

In [ ]:
# Todos los meses comprendidos entre las fechas del conjunto.
meses = pd.period_range(
    start=reservas["fecha"].min(),
    end=reservas["fecha"].max(),
    freq="M"
).astype(str)

tabla_meses = cobertura_mensual.pivot(
    index="Centro deportivo",
    columns="mes",
    values="reservas"
).reindex(columns=meses)

display(tabla_meses)

# Centros que tienen al menos un mes sin registros individuales.
centros_a_revisar = tabla_meses.loc[
    tabla_meses.isna().any(axis=1)
]

print("Centros con algún mes sin registros:", len(centros_a_revisar))
display(centros_a_revisar)

In [ ]:
cobertura_mensual.to_csv(
    carpeta_analisis / "cobertura_mensual.csv",
    index=False,
    encoding="utf-8-sig"
)

tabla_meses.to_csv(
    carpeta_analisis / "reservas_por_centro_y_mes.csv",
    encoding="utf-8-sig"
)

caso_samaranch = cobertura_mensual.loc[
    cobertura_mensual["Centro deportivo"]
    .str.contains("Samaranch", case=False, na=False)
]

display(caso_samaranch)

In [ ]:
# Seleccionar Samaranch.
samaranch = usos_padel.loc[
    usos_padel["Centro deportivo"].str.contains(
        "Samaranch", case=False, na=False
    )
].copy()

samaranch["mes"] = samaranch["fecha"].dt.to_period("M").astype("string")

# Contar registros de cada tipo por mes.
resumen_samaranch = (
    samaranch.groupby(["mes", "Tipo de ocupación"])
    .size()
    .unstack("Tipo de ocupación")
)

# Incluir todos los meses, aunque no tengan registros.
meses_revision = pd.period_range(
    reservas["fecha"].min(), reservas["fecha"].max(), freq="M"
).astype(str)

resumen_samaranch = resumen_samaranch.reindex(meses_revision)

display(resumen_samaranch)

In [ ]:
# 1. Preparar el mes en las dos tablas.
reservas["mes"] = reservas["fecha"].dt.to_period("M").astype(str)
usos_padel["mes"] = usos_padel["fecha"].dt.to_period("M").astype(str)

# 2. Crear todas las combinaciones de centro y mes.
centros = sorted(reservas["Centro deportivo"].dropna().unique())

meses = pd.period_range(
    reservas["fecha"].min(),
    reservas["fecha"].max(),
    freq="M"
).astype(str)

indice = pd.MultiIndex.from_product(
    [centros, meses],
    names=["Centro deportivo", "mes"]
)

# 3. Contar registros individuales y registros de todos los usos.
individuales = reservas.groupby(
    ["Centro deportivo", "mes"]
).size()

todos_usos = usos_padel.groupby(
    ["Centro deportivo", "mes"]
).size()

# Aquí el cero significa "cero registros encontrados en el archivo".
# No confirma que hubiera cero actividad real.
revision_cobertura = pd.DataFrame(index=indice)

revision_cobertura["registros_individuales"] = (
    individuales.reindex(indice, fill_value=0)
)

revision_cobertura["registros_todos_usos"] = (
    todos_usos.reindex(indice, fill_value=0)
)

revision_cobertura = revision_cobertura.reset_index()

# 4. Clasificar cada combinación.
revision_cobertura["situacion"] = "Con registros individuales"

sin_individuales = revision_cobertura["registros_individuales"].eq(0)
sin_ningun_uso = revision_cobertura["registros_todos_usos"].eq(0)

revision_cobertura.loc[
    sin_individuales & ~sin_ningun_uso, "situacion"
] = "Sin individuales, pero con otros usos de pádel"

revision_cobertura.loc[
    sin_ningun_uso, "situacion"
] = "Sin registros de pádel de ningún tipo"

# 5. Mostrar los meses que requieren atención.
meses_a_revisar = revision_cobertura.loc[
    sin_individuales
].copy()

print("Centros analizados:", len(centros))
print("Combinaciones centro-mes:", len(revision_cobertura))

print("\nResumen:")
print(revision_cobertura["situacion"].value_counts())

display(meses_a_revisar)

# 6. Guardar el resultado.
revision_cobertura.to_csv(
    carpeta_analisis / "revision_cobertura_centro_mes.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
revision_cobertura["anio"] = (
    revision_cobertura["mes"].str[:4].astype(int)
)

resumen_centro_anio = (
    revision_cobertura
    .assign(
        con_individuales=lambda x: x["registros_individuales"] > 0,
        con_algun_uso=lambda x: x["registros_todos_usos"] > 0
    )
    .groupby(["Centro deportivo", "anio"], as_index=False)
    .agg(
        meses_analizados=("mes", "size"),
        meses_con_individuales=("con_individuales", "sum"),
        meses_con_algun_uso=("con_algun_uso", "sum"),
        reservas_registradas=("registros_individuales", "sum")
    )
)

resumen_centro_anio["meses_sin_ningun_uso"] = (
    resumen_centro_anio["meses_analizados"]
    - resumen_centro_anio["meses_con_algun_uso"]
)

# Mostrar solo centros que tienen algún mes sin reservas individuales.
centros_con_huecos = meses_a_revisar["Centro deportivo"].unique()

display(
    resumen_centro_anio.loc[
        resumen_centro_anio["Centro deportivo"].isin(centros_con_huecos)
    ].sort_values(["Centro deportivo", "anio"])
)

resumen_centro_anio.to_csv(
    carpeta_analisis / "resumen_cobertura_por_centro_y_anio.csv",
    index=False,
    encoding="utf-8-sig"
)

## Evidencia de la equivalencia de El Quijote

In [ ]:
# Revisar las denominaciones de origen sin alterar los nombres normalizados.
nombres_origen = (usos_padel["centro_nombre_original"].astype("string")
                  .str.normalize("NFC").str.replace(r"\s+", " ", regex=True).str.strip())
quijote = usos_padel.loc[nombres_origen.isin(["CDM El Quijote", "El Quijote"])].copy()
quijote["Centro deportivo"] = nombres_origen.loc[quijote.index]
quijote["mes"] = quijote["fecha"].dt.to_period("M").astype(str)


# Comparar las unidades deportivas de cada denominación.
unidades_quijote = (
    quijote
    .groupby(
        ["Centro deportivo", "Unidad deportiva"],
        dropna=False
    )
    .size()
    .reset_index(name="registros")
)

print("UNIDADES DEPORTIVAS POR NOMBRE DEL CENTRO")
display(unidades_quijote)

# 2. Registros mensuales de todos los tipos de ocupación
meses_quijote = (
    quijote
    .groupby(["mes", "Centro deportivo"])
    .size()
    .unstack("Centro deportivo", fill_value=0)
    .sort_index()
)

print("\nREGISTROS POR MES Y NOMBRE DEL CENTRO")
display(meses_quijote)

# 3. Primera y última fecha con cada denominación
fechas_quijote = (
    quijote
    .groupby("Centro deportivo")
    .agg(
        primera_fecha=("fecha", "min"),
        ultima_fecha=("fecha", "max"),
        registros=("fecha", "size")
    )
)

print("\nINTERVALO DE FECHAS DE CADA DENOMINACIÓN")
display(fechas_quijote)

## Indicadores y visualizaciones

In [ ]:
import plotly.express as px

# 1. Asignar cada registro a una semana de lunes a domingo.
# "semana" contiene la fecha del lunes.
reservas["semana"] = (
    reservas["fecha"]
    .dt.to_period("W-SUN")
    .dt.start_time
)

# 2. Contar registros y centros representados en cada semana
resumen_semanal = (
    reservas
    .groupby("semana")
    .agg(
        reservas_registradas=("fecha", "size"),
        centros_con_registros=("Centro deportivo", "nunique"),
        dias_con_registros=("fecha", "nunique")
    )
    .sort_index()
)

# 3. Incluir todas las semanas del intervalo.
# Las semanas sin registros se dejan vacías, no se rellenan con cero.
todas_las_semanas = pd.date_range(
    start=reservas["semana"].min(),
    end=reservas["semana"].max(),
    freq="W-MON"
)

resumen_semanal = (
    resumen_semanal
    .reindex(todas_las_semanas)
    .rename_axis("semana")
    .reset_index()
)

resumen_semanal["fin_semana"] = (
    resumen_semanal["semana"] + pd.Timedelta(days=6)
)

# 4. Marcar las semanas que quedan dentro del intervalo observado.
# Evitamos que los extremos parciales parezcan caídas de reservas.
resumen_semanal["dentro_intervalo_observado"] = (
    (resumen_semanal["semana"] >= reservas["fecha"].min())
    & (resumen_semanal["fin_semana"] <= reservas["fecha"].max())
)

semanas_grafico = resumen_semanal.loc[
    resumen_semanal["dentro_intervalo_observado"]
].copy()

# 5. Crear el gráfico interactivo
fig = px.line(
    semanas_grafico,
    x="semana",
    y="reservas_registradas",
    markers=True,
    title="Evolución semanal de las reservas individuales de pádel",
    labels={
        "semana": "Semana — fecha del lunes",
        "reservas_registradas": "Registros de reservas individuales",
        "centros_con_registros": "Centros con registros",
        "dias_con_registros": "Días con registros"
    },
    hover_data=["centros_con_registros", "dias_con_registros"],
    template="plotly_white"
)

fig.update_traces(connectgaps=False)
fig.update_layout(hovermode="x unified")
fig.update_yaxes(rangemode="tozero")
fig.show()

# 6. Guardar la tabla completa y el gráfico
resumen_semanal.to_csv(
    carpeta_analisis / "reservas_semanales.csv",
    index=False,
    encoding="utf-8-sig"
)

fig.write_html(
    str(carpeta_analisis / "evolucion_semanal.html"),
    include_plotlyjs=True
)

In [ ]:
# Complemento del gráfico de reservas:
# número de centros con registros individuales cada semana.

fig_centros = px.line(
    semanas_grafico,
    x="semana",
    y="centros_con_registros",
    markers=True,
    title="Centros con registros individuales por semana",
    labels={
        "semana": "Semana — fecha del lunes",
        "centros_con_registros": "Centros con registros"
    },
    template="plotly_white"
)

fig_centros.update_traces(connectgaps=False)

fig_centros.update_yaxes(
    range=[0, reservas["Centro deportivo"].nunique() + 1],
    dtick=5
)

fig_centros.update_layout(hovermode="x unified")
fig_centros.show()

fig_centros.write_html(
    str(carpeta_analisis / "centros_con_registros_semanales.html"),
    include_plotlyjs=True
)

In [ ]:
# 1. Preparar fecha y hora de inicio
datos_horarios = reservas.copy()

datos_horarios["fecha_dia"] = (
    datos_horarios["fecha"].dt.normalize()
)

# Agrupar los inicios en intervalos de una hora.
# Ejemplo: 18:00 y 18:30 se incluyen en la franja de las 18.
datos_horarios["hora"] = (
    pd.to_numeric(datos_horarios["hora_inicio_num"], errors="raise")
    // 1
).astype(int)

# 2. Contar los registros por fecha y hora
registros_por_dia_hora = (
    datos_horarios
    .groupby(["fecha_dia", "hora"])
    .size()
    .unstack("hora", fill_value=0)
    .reindex(columns=range(24), fill_value=0)
)

# Solo aparecen fechas con algún registro individual.
# Dentro de esas fechas, cero significa que no hay registros
# que comiencen en esa franja horaria.

# 3. Calcular la media para cada día de la semana
mapa_horario = (
    registros_por_dia_hora
    .groupby(registros_por_dia_hora.index.day_of_week)
    .mean()
    .reindex(range(7))
)

dias = [
    "Lunes", "Martes", "Miércoles", "Jueves",
    "Viernes", "Sábado", "Domingo"
]

mapa_horario.index = dias
mapa_horario.columns = [f"{hora:02d}:00" for hora in range(24)]

# 4. Mostrar cuántas fechas se utilizan para cada media
dias_observados = (
    datos_horarios[["fecha_dia"]]
    .drop_duplicates()
    .assign(dia_semana=lambda x: x["fecha_dia"].dt.day_of_week)
    .groupby("dia_semana")
    .size()
    .reindex(range(7), fill_value=0)
)

dias_observados.index = dias

print("Número de fechas utilizadas para cada día de la semana:")
display(dias_observados.rename("fechas_observadas").to_frame())

# 5. Dibujar el mapa de calor
fig_horario = px.imshow(
    mapa_horario,
    aspect="auto",
    color_continuous_scale="Blues",
    labels={
        "x": "Hora de inicio de la reserva",
        "y": "Día de la semana",
        "color": "Media de registros"
    },
    title="Reservas por día y hora: media por día observado",
    template="plotly_white"
)

fig_horario.update_traces(
    hovertemplate=(
        "%{y}, franja de las %{x}"
        "<br>Media de registros: %{z:.1f}"
        "<extra></extra>"
    )
)

fig_horario.show()

# 6. Guardar resultados
mapa_horario.to_csv(
    carpeta_analisis / "media_reservas_dia_hora.csv",
    index_label="dia_semana",
    encoding="utf-8-sig"
)

fig_horario.write_html(
    str(carpeta_analisis / "mapa_calor_dia_hora.html"),
    include_plotlyjs=True
)

In [ ]:
# Variación respecto a la semana anterior
comparacion_semanal = (
    semanas_grafico
    .sort_values("semana")
    .copy()
)

# Recuperar los valores de la semana anterior
comparacion_semanal["reservas_semana_anterior"] = (
    comparacion_semanal["reservas_registradas"].shift(1)
)

comparacion_semanal["centros_semana_anterior"] = (
    comparacion_semanal["centros_con_registros"].shift(1)
)

# Evitar dividir entre cero.
# Si falta el dato anterior, la variación queda vacía.
base_comparacion = comparacion_semanal[
    "reservas_semana_anterior"
].where(
    comparacion_semanal["reservas_semana_anterior"] > 0
)

comparacion_semanal["variacion_pct"] = (
    (
        comparacion_semanal["reservas_registradas"]
        / base_comparacion
    ) - 1
) * 100

comparacion_semanal["cambio_numero_centros"] = (
    comparacion_semanal["centros_con_registros"]
    - comparacion_semanal["centros_semana_anterior"]
)

columnas_mostrar = [
    "semana",
    "reservas_registradas",
    "reservas_semana_anterior",
    "variacion_pct",
    "centros_con_registros",
    "cambio_numero_centros"
]

display(
    comparacion_semanal[columnas_mostrar]
    .tail(12)
    .round({"variacion_pct": 2})
)

comparacion_semanal.to_csv(
    carpeta_analisis / "comparacion_semanal.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
# Preparar los datos que utilizará la aplicación Dash.
# "reservas" ya debe tener los nombres de centros normalizados.

carpeta_app = Path("Datos/app")
carpeta_app.mkdir(parents=True, exist_ok=True)

# Conservar únicamente las columnas necesarias para la aplicación.
datos_app = reservas[
    [
        "fecha",
        "Centro deportivo",
        "hora_inicio_num",
        "duracion_minutos"
    ]
].copy()

datos_app = datos_app.sort_values(
    ["fecha", "Centro deportivo", "hora_inicio_num"]
)

archivo_app = carpeta_app / "reservas_padel_app.csv"

datos_app.to_csv(
    archivo_app,
    index=False,
    encoding="utf-8-sig"
)

print("Registros exportados:", len(datos_app))
print("Centros:", datos_app["Centro deportivo"].nunique())
print("Primera fecha:", datos_app["fecha"].min())
print("Última fecha:", datos_app["fecha"].max())
print("Tamaño del archivo (MB):", round(
    archivo_app.stat().st_size / (1024 ** 2), 2
))
print("Archivo guardado en:", archivo_app.resolve())

## Conclusiones del análisis exploratorio

### 1. Datos analizados y normalización de centros

El conjunto analizado contiene 417.661 registros de reservas individuales,
con fechas comprendidas entre el 2 de enero de 2025 y el 31 de agosto de 2026:
256.122 corresponden a 2025 y 161.539 a 2026.

La normalización de los nombres ha reducido de 48 a 45 las denominaciones
distintas de centros deportivos, sin eliminar registros. Se corrigieron
espacios sobrantes y se unificaron «CDM El Quijote» y «El Quijote», al
observar unidades deportivas coincidentes y periodos mensuales
complementarios. Se conservaron los nombres originales para mantener
la trazabilidad de los cambios.

Los totales anuales no son directamente comparables, ya que 2025 incluye
doce meses y 2026 únicamente ocho.

### 2. Evolución semanal y cobertura

La serie semanal muestra una disminución de registros durante el verano
en ambos años, especialmente en agosto. En 2025 se observa una
recuperación posterior.

Este descenso coincide con una reducción del número de centros con
registros individuales: frente a aproximadamente 41–44 centros durante
buena parte del periodo, algunas semanas de agosto presentan alrededor
de 22–25.

Por tanto, la caída del total no puede atribuirse exclusivamente a una
menor demanda. También cambia el conjunto de centros representados.
Los datos disponibles no permiten distinguir por sí solos entre cierres,
ausencia de reservas y falta de información.

### 3. Distribución por día de la semana y hora

El mapa de calor muestra una mayor concentración de inicios de reservas
por la tarde entre semana. Durante los fines de semana destacan también
las franjas de la mañana.

El indicador representa la media de registros por franja horaria y día
observado para el conjunto de centros. Solo se incluyen fechas con
algún registro individual y se combinan ambos años.

Cada registro se asigna a su hora de inicio. Por ello, el mapa describe
cuándo comienzan las reservas, no cuántas pistas permanecen ocupadas
durante cada hora.

### 4. Variación respecto a la semana anterior

La variación porcentual permite identificar aumentos y disminuciones
del volumen registrado entre semanas consecutivas.

Por ejemplo, en la semana del 3 de agosto de 2026 los registros disminuyen
un 16,49 % respecto a la anterior, mientras que los centros con registros
pasan de 39 a 24. En la semana del 24 de agosto aumentan un 33,70 %,
manteniéndose en 25 el número de centros con registros.

Estos ejemplos muestran la necesidad de interpretar conjuntamente
el volumen y la cobertura. Un mismo número de centros no garantiza
que sean los mismos ni que aporten información igualmente completa.

### 5. Limitaciones y aplicación al proyecto

La ausencia de registros no se interpreta automáticamente como ausencia
de demanda ni como cierre de una instalación. Asimismo, encontrar
registros en un mes no garantiza que su información esté completa.

Sin datos sobre pistas disponibles, horarios de apertura y cierres,
no puede calcularse un porcentaje real de ocupación. Tampoco se observa
la demanda no satisfecha, como intentos de reserva sin disponibilidad.

Los tres indicadores desarrollados —volumen semanal, patrón por día
y hora y variación semanal— constituyen la base de la aplicación Dash.
Su interpretación deberá acompañarse del periodo seleccionado y de
información sobre la cobertura de los datos.